In [1]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("eda").getOrCreate()

from pyspark.sql.types import *

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
schema = StructType([
    StructField('time',IntegerType(),True),
    StructField('src_user',StringType(),True),
    StructField('dest_user',StringType(),True),
    StructField('src_comp',StringType(),True),
    StructField('dest_comp',StringType(),True),
    StructField('auth_type',StringType(),True),
    StructField('logon_type',StringType(),True),
    StructField('auth_orientation',StringType(),True),
    StructField('outcome',StringType(),True),
])

In [ ]:
df = spark.read.csv('/content/drive/MyDrive/auth.txt.gz', header=False, schema=schema)
df.show(5)

+----+--------------------+--------------------+--------+---------+---------+----------+----------------+-------+
|time|            src_user|           dest_user|src_comp|dest_comp|auth_type|logon_type|auth_orientation|outcome|
+----+--------------------+--------------------+--------+---------+---------+----------+----------------+-------+
|   1|ANONYMOUS LOGON@C586|ANONYMOUS LOGON@C586|    C586|     C586|        ?|   Network|          LogOff|Success|
|   1|          C101$@DOM1|          C101$@DOM1|    C988|     C988|        ?|   Network|          LogOff|Success|
|   1|         C1020$@DOM1|        SYSTEM@C1020|   C1020|    C1020|Negotiate|   Service|           LogOn|Success|
|   1|         C1021$@DOM1|         C1021$@DOM1|   C1021|     C625| Kerberos|   Network|           LogOn|Success|
|   1|         C1035$@DOM1|         C1035$@DOM1|   C1035|     C586| Kerberos|   Network|           LogOn|Success|
+----+--------------------+--------------------+--------+---------+---------+----------+

In [5]:
from pyspark.sql.functions import countDistinct

## Count rows
print(df.count())

# Distinct number of source computers 
print(df.select(countDistinct("src_comp")).collect()[0][0])

# Distinct number of destination computers
print(df.select(countDistinct("dest_comp")).collect()[0][0])

1051430458


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
from pyspark.sql.functions import col, contains

# Machine count
print(df.filter(col('src_user').contains('$')).count())

## Proportion of machines
print(df.filter(col('src_user').contains('$')).count()/df.count())

In [ ]:
# Number of success & failures 
df.groupBy('outcome').count().show()

In [ ]:
# %%
# Number of success and failures for machines
df.filter(col('src_user').contains('$')).groupBy('outcome').count().show()

In [ ]:
# Distribution of human source users
df.filter(col('src_user').startswith('U')).groupBy('src_user').count().orderBy('count',ascending = False).show()

In [ ]:
# Distribution of destination users
df.groupBy("dest_user").count().orderBy("count", ascending=False).show(20)